# 01 · Bronze layer: load landing CSVs into Delta tables

**Project:** IPL data engineering (Cricsheet ball-by-ball data, 2008–2026)
**Layer:** bronze. The 11 CSVs produced by the Python pipeline (`data/processed`) are loaded **1:1** into `workspace.bronze.<table>`. No cleaning and no business logic here; validation and the star schema belong to silver.

**Pattern used for every table:** inspect the file (`LIMIT 5`, `DESCRIBE QUERY`) → declare the schema → `CREATE OR REPLACE TABLE ... AS SELECT * FROM read_files(...)` → verify the load.

**Conventions**
- **Declared schemas, not inferred.** Inference is a guess (for example `match_id` looks numeric, so it is inferred as `INT`) and adds a `_rescued_data` column.
- **Ids from outside** (`match_id`, player ids) are `STRING`. **Ids we generate** (season, team, venue, `date_key`) are `INT`.
- The schema lists columns in the **same order as the file**.
- A value that does not fit a declared type becomes **NULL silently**, so every load is followed by a check.
- `CREATE OR REPLACE` makes each cell safe to rerun.

| Table | Rows loaded |
|---|---|
| dim_season | 19 |
| dim_team | 15 |
| dim_venue | 36 |
| dim_date | 6,618 |
| venue_alias | 60 |
| team_alias | 19 |
| season_alias | 19 |
| fact_match | 1,243 |
| dim_player | 964 |
| bridge_wicket_fielder | 11,266 |
| fact_delivery | 295,732 |

## 0 · Setup: volume, schema, landing check
The volume `workspace.default.ipl_landing` holds the 11 uploaded CSVs. The bronze schema holds the tables built below.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.ipl_landing;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;

In [0]:
%sql
LIST '/Volumes/workspace/default/ipl_landing/';

path,name,size,modification_time
/Volumes/workspace/default/ipl_landing/bridge_wicket_fielder.csv,bridge_wicket_fielder.csv,324560,1794051238000
/Volumes/workspace/default/ipl_landing/dim_date.csv,dim_date.csv,296624,1794051238000
/Volumes/workspace/default/ipl_landing/dim_player.csv,dim_player.csv,20407,1794051238000
/Volumes/workspace/default/ipl_landing/dim_season.csv,dim_season.csv,185,1794051238000
/Volumes/workspace/default/ipl_landing/dim_team.csv,dim_team.csv,344,1794051238000
/Volumes/workspace/default/ipl_landing/dim_venue.csv,dim_venue.csv,1143,1794051238000
/Volumes/workspace/default/ipl_landing/fact_delivery.csv,fact_delivery.csv,21077941,1794051245000
/Volumes/workspace/default/ipl_landing/fact_match.csv,fact_match.csv,39866,1794051238000
/Volumes/workspace/default/ipl_landing/season_alias.csv,season_alias.csv,195,1794051238000
/Volumes/workspace/default/ipl_landing/team_alias.csv,team_alias.csv,447,1794051238000


## 1 · dim_season
Inspect the file, then declare the schema (`INT`, `INT`) and load.

In [0]:
%sql
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/dim_season.csv',
    format => 'csv',
    header => true
);

season_id,season_year,_rescued_data
1,2008,null
2,2009,null
3,2010,null
4,2011,null
5,2012,null
6,2013,null
7,2014,null
8,2015,null
9,2016,null
10,2017,null


In [0]:
%sql
DESCRIBE QUERY
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/dim_season.csv',
    format => 'csv',
    header => true
);

col_name,data_type,comment,metadata
season_id,int,null,null
season_year,int,null,null
_rescued_data,string,null,null


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.dim_season AS
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/dim_season.csv',
    format => 'csv',
    header => true,
    schema => 'season_id INT, season_year INT'
);

num_affected_rows,num_inserted_rows


Verify the stored column types and the row count (expected 19).

In [0]:
%sql
DESCRIBE TABLE workspace.bronze.dim_season;

col_name,data_type,comment
season_id,int,null
season_year,int,null
# Clustering Information,,
# col_name,data_type,comment


In [0]:
%sql
SELECT count(*) FROM workspace.bronze.dim_season;

count(*)
19


## 2 · fact_match
`match_id` is the Cricsheet id from outside, so it is `STRING`. Everything else is a generated id, so `INT`.

`winner_team_id` is legitimately NULL for matches without a winner. The pre-load check below shows 1,243 rows with 1,218 winners, so **25 matches have no winner** (no-result matches, and ties decided by a super over, where the result is stored in `eliminator_team_id`).

In [0]:
%sql
DESCRIBE QUERY
SELECT * FROM read_files(
  '/Volumes/workspace/default/ipl_landing/fact_match.csv',
  format => 'csv',
  header => true
);

col_name,data_type,comment,metadata
match_id,int,null,null
season_id,int,null,null
venue_id,int,null,null
team1_id,int,null,null
team2_id,int,null,null
winner_team_id,int,null,null
result,string,null,null
eliminator_team_id,int,null,null
date_key,int,null,null
_rescued_data,string,null,null


In [0]:
%sql
SELECT * FROM read_files(
  '/Volumes/workspace/default/ipl_landing/fact_match.csv',
  format => 'csv',
  header => true
) LIMIT 5;

match_id,season_id,venue_id,team1_id,team2_id,winner_team_id,result,eliminator_team_id,date_key,_rescued_data
1082591,10,13,15,14,15,null,null,20170405,null
1082592,10,11,13,10,13,null,null,20170406,null
1082593,10,32,4,8,8,null,null,20170407,null
1082594,10,25,6,13,6,null,null,20170408,null
1082595,10,4,14,3,14,null,null,20170408,null


In [0]:
%sql
SELECT count(*) AS total_rows,
       count(winner_team_id) AS rows_with_winner
FROM read_files(
  '/Volumes/workspace/default/ipl_landing/fact_match.csv',
  format => 'csv',
  header => true,
  schema => 'match_id STRING, season_id INT, venue_id INT, team1_id INT, team2_id INT, winner_team_id INT, result STRING, eliminator_team_id INT, date_key INT'
);

total_rows,rows_with_winner
1243,1218


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.fact_match AS
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/fact_match.csv',
    format => 'csv',
    header => true,
    schema => 'match_id STRING, season_id INT, venue_id INT, team1_id INT, team2_id INT, winner_team_id INT, result STRING, eliminator_team_id INT, date_key INT'
);

num_affected_rows,num_inserted_rows


Verify the row count (expected 1,243).

In [0]:
%sql
SELECT count(*) AS total_rows FROM workspace.bronze.fact_match;

total_rows
1243


## 3 · dim_team

In [0]:
%sql
DESCRIBE QUERY
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/dim_team.csv',
    format => 'csv',
    header => true
);

col_name,data_type,comment,metadata
team_id,int,null,null
team_name,string,null,null
_rescued_data,string,null,null


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.dim_team AS
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/dim_team.csv',
    format => 'csv',
    header => true,
    schema => 'team_id INT, team_name STRING'
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT count(*) AS total_rows FROM workspace.bronze.dim_team;

total_rows
15


## 4 · dim_venue

In [0]:
%sql
DESCRIBE QUERY
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/dim_venue.csv',
    format => 'csv',
    header => true
);

col_name,data_type,comment,metadata
venue_id,int,null,null
venue_name,string,null,null
_rescued_data,string,null,null


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.dim_venue AS
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/dim_venue.csv',
    format => 'csv',
    header => true,
    schema => 'venue_id INT, venue_name STRING'
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT count(*) as total_rows FROM workspace.bronze.dim_venue;

total_rows
36


## 5 · dim_date
Contains a `DATE` column and a `BOOLEAN` flag, so the declared types matter here.

In [0]:
%sql
DESCRIBE QUERY
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/dim_date.csv',
    format => 'csv',
    header => true
);

col_name,data_type,comment,metadata
date_key,int,null,null
full_date,date,null,null
year,int,null,null
month,int,null,null
day,int,null,null
weekday_name,string,null,null
is_weekend,boolean,null,null
_rescued_data,string,null,null


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.dim_date AS
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/dim_date.csv',
    format => 'csv',
    header => true,
    schema => 'date_key INT, full_date DATE, year INT, month INT, day INT, weekday_name STRING, is_weekend BOOLEAN'
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT count(*) AS total_rows FROM workspace.bronze.dim_date;

total_rows
6618


## 6 · venue_alias
Raw venue spelling → generated `venue_id`.

In [0]:
%sql
DESCRIBE QUERY
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/venue_alias.csv',
    format => 'csv',
    header => true
);

col_name,data_type,comment,metadata
venue_spelling,string,null,null
venue_id,int,null,null
_rescued_data,string,null,null


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.venue_alias AS
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/venue_alias.csv',
    format => 'csv',
    header => true,
    schema => 'venue_spelling STRING, venue_id INT'
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT count(*) AS total_rows FROM workspace.bronze.venue_alias;

total_rows
60


## 7 · team_alias
Raw team spelling → generated `team_id`.

In [0]:
%sql
DESCRIBE QUERY
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/team_alias.csv',
    format => 'csv',
    header => true
);

col_name,data_type,comment,metadata
team_spelling,string,null,null
team_id,int,null,null
_rescued_data,string,null,null


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.team_alias AS
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/team_alias.csv',
    format => 'csv',
    header => true,
    schema => 'team_spelling STRING, team_id INT'
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT count(*) AS total_rows FROM workspace.bronze.team_alias;

total_rows
19


## 8 · season_alias
Raw season label → generated `season_id`.

In [0]:
%sql
DESCRIBE QUERY
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/season_alias.csv',
    format => 'csv',
    header => true
);

col_name,data_type,comment,metadata
season_label,string,null,null
season_id,int,null,null
_rescued_data,string,null,null


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.season_alias AS
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/season_alias.csv',
    format => 'csv',
    header => true,
    schema => 'season_label STRING, season_id INT'
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT count(*) AS total_rows FROM workspace.bronze.season_alias;

total_rows
19


## 9 · dim_player
Player ids are hex strings from outside (for example `18e6906e`), so `player_id` must be `STRING`.

In [0]:
%sql
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/dim_player.csv',
    format => 'csv',
    header => true
) LIMIT 5;

player_id,player_name,_rescued_data
18e6906e,A Choudhary,null
6fda55cc,A Deshmukh,null
96fd40ae,A Nehra,null
2efc430e,AY Dandekar,null
2e81a32d,B Kumar,null


In [0]:
%sql
DESCRIBE QUERY
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/dim_player.csv',
    format => 'csv',
    header => true
);

col_name,data_type,comment,metadata
player_id,string,null,null
player_name,string,null,null
_rescued_data,string,null,null


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.dim_player AS
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/dim_player.csv',
    format => 'csv',
    header => true,
    schema => 'player_id STRING, player_name STRING'
);

num_affected_rows,num_inserted_rows


Verify the row count (expected 964).

In [0]:
%sql
SELECT count(*) AS total_rows from workspace.bronze.dim_player;

total_rows
964


### Demonstration: why `player_id` must not be `INT`
This is a read-only `SELECT`; it does **not** write to bronze.

Declaring `player_id INT` does **not** raise an error. Values that cannot be cast quietly become NULL: only 19 of 964 ids survive (the ones made of digits only) and 945 become NULL. Every later join on `player_id` would silently lose about 98% of players.

In [0]:
%sql
SELECT
  COUNT(*)                                     AS total_rows,
  COUNT(player_id)                             AS non_null_ids,
  COUNT(*) - COUNT(player_id)                  AS null_ids
FROM read_files(
  '/Volumes/workspace/default/ipl_landing/dim_player.csv',
  format => 'csv',
  header => true,
  schema => 'player_id INT, player_name STRING'
);

total_rows,non_null_ids,null_ids
964,19,945


## 10 · bridge_wicket_fielder
Links a wicket to the fielder(s) involved. Inference gives `match_id` as `INT`; it is declared `STRING` to match `fact_match.match_id`.

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/default/ipl_landing/bridge_wicket_fielder.csv',
  format => 'csv',
  header => true
)
LIMIT 5;

match_id,innings_number,delivery_seq,fielder_id,is_substitute,_rescued_data
1082591,1,12,c3a96caf,false,null
1082591,1,65,dc9dd038,false,null
1082591,1,95,dc9dd038,false,null
1082591,2,41,dcce6f09,false,null
1082591,2,75,2e11c706,false,null


In [0]:
%sql
DESCRIBE QUERY
SELECT *
FROM read_files(
  '/Volumes/workspace/default/ipl_landing/bridge_wicket_fielder.csv',
  format => 'csv',
  header => true
);

col_name,data_type,comment,metadata
match_id,int,null,null
innings_number,int,null,null
delivery_seq,int,null,null
fielder_id,string,null,null
is_substitute,boolean,null,null
_rescued_data,string,null,null


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.bridge_wicket_fielder AS
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/bridge_wicket_fielder.csv',
    format => 'csv',
    header => true,
    schema => 'match_id STRING, innings_number INT, delivery_seq INT, fielder_id STRING, is_substitute BOOLEAN'
);

num_affected_rows,num_inserted_rows


Verify the row count (expected 11,266).

In [0]:
%sql
SELECT count(*) AS total_rows FROM workspace.bronze.bridge_wicket_fielder;

total_rows
11266


Null check: compare `COUNT(*)` with `COUNT(column)` for every column. The columns at risk are the ones declared with a stricter type (`INT`, `BOOLEAN`), because a failed cast becomes NULL silently. All six numbers equal 11,266, so nothing was dropped.

In [0]:
%sql
SELECT count(*) AS total_rows,
count(match_id) AS match_id,
count(innings_number) AS innings_number,
count(delivery_seq) AS delivery_seq,
count(fielder_id) AS fielder_id,
count(is_substitute) AS is_substitute
FROM workspace.bronze.bridge_wicket_fielder;

total_rows,match_id,innings_number,delivery_seq,fielder_id,is_substitute
11266,11266,11266,11266,11266,11266


## 11 · fact_delivery
Grain: one row per delivery. The schema below lists the 16 columns **in file order** (taken from `DESCRIBE QUERY`). `match_id` is overridden from the inferred `INT` to `STRING`; the four flags are `BOOLEAN`.

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/default/ipl_landing/fact_delivery.csv',
  format => 'csv',
  header => true
)
LIMIT 5;

match_id,innings_number,delivery_seq,is_legal_delivery,runs_batter,runs_extras,runs_total,is_wicket,wicket_kind,is_bowler_wicket,batter_id,bowler_id,non_striker_id,batting_team_id,over_number,player_out_id,_rescued_data
1082591,1,1,true,0,0,0,false,null,false,dcce6f09,245c97cb,0a476045,15,1,null,null
1082591,1,2,true,0,0,0,false,null,false,dcce6f09,245c97cb,0a476045,15,1,null,null
1082591,1,3,true,4,0,4,false,null,false,dcce6f09,245c97cb,0a476045,15,1,null,null
1082591,1,4,true,0,0,0,false,null,false,dcce6f09,245c97cb,0a476045,15,1,null,null
1082591,1,5,false,0,2,2,false,null,false,dcce6f09,245c97cb,0a476045,15,1,null,null


In [0]:
%sql
DESCRIBE QUERY
SELECT *
FROM read_files(
  '/Volumes/workspace/default/ipl_landing/fact_delivery.csv',
  format => 'csv',
  header => true
);

col_name,data_type,comment,metadata
match_id,int,null,null
innings_number,int,null,null
delivery_seq,int,null,null
is_legal_delivery,boolean,null,null
runs_batter,int,null,null
runs_extras,int,null,null
runs_total,int,null,null
is_wicket,boolean,null,null
wicket_kind,string,null,null
is_bowler_wicket,boolean,null,null


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.bronze.fact_delivery AS
SELECT * FROM read_files(
    '/Volumes/workspace/default/ipl_landing/fact_delivery.csv',
    format => 'csv',
    header => true,
    schema => 'match_id STRING, innings_number INT, delivery_seq INT, is_legal_delivery BOOLEAN, runs_batter INT, runs_extras INT, runs_total INT, is_wicket BOOLEAN, wicket_kind STRING, is_bowler_wicket BOOLEAN, batter_id STRING, bowler_id STRING, non_striker_id STRING, batting_team_id INT, over_number INT, player_out_id STRING'
);

num_affected_rows,num_inserted_rows


Null check on all 16 columns. Every column is fully populated (295,732) except `wicket_kind` and `player_out_id`, which are legitimately NULL on non-wicket deliveries. Both show **14,705**, which equals the wicket total from the Python pipeline.

In [0]:
%sql
SELECT count(*) AS total_rows,
count(match_id) AS match_id,
count(innings_number) AS innings_number,
count(delivery_seq) AS delivery_seq,
count(is_legal_delivery) AS is_legal_delivery,
count(runs_batter) AS runs_batter,
count(runs_extras) AS runs_extras,
count(runs_total) AS runs_total,
count(is_wicket) AS is_wicket,
count(wicket_kind) AS wicket_kind,
count(is_bowler_wicket) AS is_bowler_wicket,
count(batter_id) AS batter_id,
count(bowler_id) AS bowler_id,
count(non_striker_id) AS non_striker_id,
count(batting_team_id) AS batting_team_id,
count(over_number) AS over_number,
count(player_out_id) AS player_out_id FROM workspace.bronze.fact_delivery;

total_rows,match_id,innings_number,delivery_seq,is_legal_delivery,runs_batter,runs_extras,runs_total,is_wicket,wicket_kind,is_bowler_wicket,batter_id,bowler_id,non_striker_id,batting_team_id,over_number,player_out_id
295732,295732,295732,295732,295732,295732,295732,295732,295732,14705,295732,295732,295732,295732,295732,295732,14705


## Bronze complete
All 11 tables are loaded and count-verified. The checks above prove that no values were dropped by type casts; they do **not** prove the values are correct. Value-level validation (run totals, wicket totals, orphan checks) is done in the **silver** layer.